# 00 — Setup & Verify

**Purpose:** Sanity-check the environment and data integrity before any analysis begins.

This notebook does **not** run benchmarks. It only reads from:
- `results/raw/` — benchmark result CSVs
- `data/benchmark_real/` — real dataset (Amazon Reviews)
- `data/benchmark_syn/` — synthetic dataset

---

**Checks performed:**
1. Library versions
2. Directory structure
3. Result file discovery
4. Schema consistency
5. Missing values & invalid status
6. Expected labels (framework / workload / dataset_size)
7. Run count completeness
8. Data folder contents
9. Summary statistics preview

## 1. Library Versions

In [1]:
import pandas as pd
import polars as pl
import dask
import numpy as np
import pyarrow as pa
import psutil
import sys
from pathlib import Path

print(f"Python      : {sys.version.split()[0]}")
print(f"pandas      : {pd.__version__}")
print(f"polars      : {pl.__version__}")
print(f"dask        : {dask.__version__}")
print(f"numpy       : {np.__version__}")
print(f"pyarrow     : {pa.__version__}")
print(f"psutil      : {psutil.__version__}")

ram_gb = psutil.virtual_memory().total / 1e9
print(f"\nSystem RAM  : {ram_gb:.1f} GB")

Python      : 3.12.3
pandas      : 2.2.2
polars      : 0.20.31
dask        : 2024.5.0
numpy       : 1.26.4
pyarrow     : 15.0.2
psutil      : 5.9.8

System RAM  : 16.8 GB


## 2. Directory Structure

In [2]:
# ── Edit these paths if your project root differs ──
PROJECT_ROOT = Path("..")  # adjust relative to notebook location
RESULTS_DIR  = PROJECT_ROOT / "results" / "raw"
REAL_DIR     = PROJECT_ROOT / "data" / "benchmark_real"
SYN_DIR      = PROJECT_ROOT / "data" / "benchmark_syn"

dirs = {
    "results/raw"         : RESULTS_DIR,
    "data/benchmark_real" : REAL_DIR,
    "data/benchmark_syn"  : SYN_DIR,
}

all_ok = True
for label, path in dirs.items():
    exists = path.exists()
    status = "OK" if exists else "MISSING"
    print(f"{status}  {label}  →  {path.resolve()}")
    if not exists:
        all_ok = False

if not all_ok:
    print("\n One or more directories are missing. Fix paths before continuing.")
else:
    print("\nAll required directories found.")

OK  results/raw  →  D:\Polar vs Dask\results\raw
OK  data/benchmark_real  →  D:\Polar vs Dask\data\benchmark_real
OK  data/benchmark_syn  →  D:\Polar vs Dask\data\benchmark_syn

All required directories found.


## 3. Result File Discovery

In [3]:
csv_files = sorted(RESULTS_DIR.glob("*.csv"))

print(f"Found {len(csv_files)} CSV file(s) in results/raw/\n")
for f in csv_files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45}  {size_kb:>8.1f} KB")

if not csv_files:
    print("No result files found — cannot proceed.")

Found 12 CSV file(s) in results/raw/

  dask_real_results.csv                               3.4 KB
  dask_size_results.csv                               3.5 KB
  dask_syn_results.csv                                3.4 KB
  pandas_real_results.csv                             2.8 KB
  pandas_size_results.csv                             2.9 KB
  pandas_syn_results.csv                              2.8 KB
  polars_eager_real_results.csv                       3.0 KB
  polars_eager_size_results.csv                       3.1 KB
  polars_eager_syn_results.csv                        3.0 KB
  polars_real_results.csv                             3.0 KB
  polars_size_results.csv                             3.0 KB
  polars_syn_results.csv                              3.0 KB


## 4. Load & Inspect Schema

Expected columns:

| Column | Type | Description |
|---|---|---|
| `timestamp` | str | ISO-8601 run timestamp |
| `framework` | str | `pandas`, `polars_eager`, `polars_lazy`, `dask` |
| `workload` | str | `filter`, `groupby`, `join`, `pipeline` |
| `dataset_size` | str | `1M`, `10M`, `50M` (Track A) or `1GB`, `5GB`, `20GB` (Track B) |
| `n_rows` | int | Actual row count processed |
| `run_index` | int | 1, 2, or 3 |
| `time_s` | float | Wall-clock execution time (seconds) |
| `peak_memory_mb` | float | Peak RSS memory (MB) |
| `throughput_rows_per_s` | int | rows / time_s |
| `status` | str | `ok`, `error`, `oom`, `timeout` |
| `notes` | str | Optional metadata (e.g. partition config) |

In [11]:
EXPECTED_COLUMNS = [
    "timestamp", "framework", "workload", "dataset_size",
    "n_rows", "run_index", "time_s", "peak_memory_mb",
    "throughput_rows_per_s", "status", "notes"
]

frames = []
schema_issues = []

for f in csv_files:
    df = pd.read_csv(f)
    missing_cols = set(EXPECTED_COLUMNS) - set(df.columns)
    extra_cols   = set(df.columns) - set(EXPECTED_COLUMNS)
    
    if missing_cols:
        schema_issues.append(f" ERROR {f.name}: missing columns → {missing_cols}")
    if extra_cols:
        schema_issues.append(f" WARNING  {f.name}: extra columns   → {extra_cols}")
    
    df["_source_file"] = f.name
    frames.append(df)


if schema_issues:
    print("Schema issues detected:")
    for issue in schema_issues:
        print(issue)
else:
    print("All files have the expected schema.")

# Combine all results
results = pd.concat(frames, ignore_index=True)
print(f"\nTotal rows loaded: {len(results):,}")
results.sample(5)

All files have the expected schema.

Total rows loaded: 432


,timestamp,framework,workload,dataset_size,n_rows,run_index,time_s,peak_memory_mb,throughput_rows_per_s,status,notes,_source_file
293,2026-04-22T23:15:57,polars_eager,groupby,1M,1000000,3,0.5337,802.11,1873575,ok,NaN,polars_eager_syn_results.csv
324,2026-04-21T10:57:08,polars_lazy,filter,1M,1000000,1,0.3976,699.18,2515399,ok,NaN,polars_real_results.csv
223,2026-04-21T11:02:13,polars_eager,join,1M,1000000,2,1.5992,3888.21,625296,ok,NaN,polars_eager_real_results.csv
151,2026-04-23T12:17:59,pandas,join,5GB,6777185,2,36.6398,8036.76,184968,ok,NaN,pandas_size_results.csv
111,2026-04-21T10:46:02,pandas,groupby,1M,1000000,1,3.1031,1384.84,322258,ok,NaN,pandas_real_results.csv


## 5. Missing Values & Status Check

In [12]:
# ── Missing values (excluding `notes` which is intentionally sparse) ──
critical_cols = [c for c in EXPECTED_COLUMNS if c != "notes"]
null_counts = results[critical_cols].isnull().sum()
null_counts = null_counts[null_counts > 0]

if null_counts.empty:
    print("OK - No missing values in critical columns.")
else:
    print("ERROR - Missing values found:")
    print(null_counts.to_string())

print()

# ── Status distribution ──
status_counts = results["status"].value_counts()
print("Status distribution:")
print(status_counts.to_string())

non_ok = results[results["status"] != "ok"]
if not non_ok.empty:
    print(f"\nWARNING - {len(non_ok)} non-ok row(s) found:")
    print(non_ok[["_source_file", "framework", "workload", "dataset_size", "status", "notes"]].to_string(index=False))
else:
    print("\nAll rows have status == 'ok'.")

OK - No missing values in critical columns.

Status distribution:
status
ok    432

All rows have status == 'ok'.


## 6. Expected Labels Validation

In [13]:
EXPECTED_FRAMEWORKS  = {"pandas", "polars_eager", "polars_lazy", "dask"}
EXPECTED_WORKLOADS   = {"filter", "groupby", "join", "pipeline"}
# Track A sizes (row-based) and Track B sizes (memory-based) — never mix
TRACK_A_SIZES = {"1M", "10M", "50M"}
TRACK_B_SIZES = {"1GB", "5GB", "10GB", "20GB"}

found_frameworks = set(results["framework"].unique())
found_workloads  = set(results["workload"].unique())
found_sizes      = set(results["dataset_size"].unique())

unknown_fw   = found_frameworks - EXPECTED_FRAMEWORKS
unknown_wl   = found_workloads  - EXPECTED_WORKLOADS
unknown_sz   = found_sizes      - TRACK_A_SIZES - TRACK_B_SIZES

print(f"Frameworks  found : {sorted(found_frameworks)}")
print(f"Workloads   found : {sorted(found_workloads)}")
print(f"Dataset sizes found: {sorted(found_sizes)}")

issues = []
if unknown_fw:  issues.append(f"ERROR - Unknown framework(s) : {unknown_fw}")
if unknown_wl:  issues.append(f"ERROR - Unknown workload(s)  : {unknown_wl}")
if unknown_sz:  issues.append(f"ERROR - Unknown dataset_size : {unknown_sz}")

# Warn if Track A and Track B sizes co-exist in same file (should be separate)
for f in csv_files:
    df_f = results[results["_source_file"] == f.name]
    sizes_in_file = set(df_f["dataset_size"].unique())
    if sizes_in_file & TRACK_A_SIZES and sizes_in_file & TRACK_B_SIZES:
        issues.append(f"WARNING - {f.name}: mixes Track A and Track B sizes — {sizes_in_file}")

if issues:
    print()
    for i in issues:
        print(i)
else:
    print("\nAll labels are valid.")

Frameworks  found : ['dask', 'pandas', 'polars_eager', 'polars_lazy']
Workloads   found : ['filter', 'groupby', 'join', 'pipeline']
Dataset sizes found: ['10GB', '10M', '1M', '20GB', '50M', '5GB']

All labels are valid.


## 7. Run Count Completeness

Every `(framework, workload, dataset_size)` combination must have **exactly 3 runs** (`run_index` ∈ {1, 2, 3}).

In [14]:
ok_results = results[results["status"] == "ok"].copy()

run_counts = (
    ok_results
    .groupby(["framework", "workload", "dataset_size"])["run_index"]
    .count()
    .reset_index()
    .rename(columns={"run_index": "n_runs"})
)

incomplete = run_counts[run_counts["n_runs"] != 3]

if incomplete.empty:
    print("All (framework, workload, dataset_size) combinations have exactly 3 runs.")
else:
    print(f"WARNING - {len(incomplete)} incomplete combination(s):")
    print(incomplete.to_string(index=False))

print(f"\nTotal valid combinations: {len(run_counts)}")
run_counts.head(10)

WARNING - 48 incomplete combination(s):
   framework workload dataset_size  n_runs
        dask   filter          10M       6
        dask   filter           1M       6
        dask   filter          50M       6
        dask  groupby          10M       6
        dask  groupby           1M       6
        dask  groupby          50M       6
        dask     join          10M       6
        dask     join           1M       6
        dask     join          50M       6
        dask pipeline          10M       6
        dask pipeline           1M       6
        dask pipeline          50M       6
      pandas   filter          10M       6
      pandas   filter           1M       6
      pandas   filter          50M       6
      pandas  groupby          10M       6
      pandas  groupby           1M       6
      pandas  groupby          50M       6
      pandas     join          10M       6
      pandas     join           1M       6
      pandas     join          50M       6
      pandas p

,framework,workload,dataset_size,n_runs
0,dask,filter,10GB,3
1,dask,filter,10M,6
2,dask,filter,1M,6
3,dask,filter,20GB,3
4,dask,filter,50M,6
5,dask,filter,5GB,3
6,dask,groupby,10GB,3
7,dask,groupby,10M,6
8,dask,groupby,1M,6
9,dask,groupby,20GB,3


## 8. Data Folder Contents

Check that expected size-subfolders exist in `benchmark_real/` and `benchmark_syn/`, and that they contain parquet files.

In [15]:
REAL_EXPECTED_SIZES = ["1M", "10M", "50M"]
SYN_EXPECTED_SIZES  = ["1M", "10M", "50M", "5GB", "10GB", "20GB"]

def check_data_folder(root: Path, expected_sizes: list):
    print(f"\nFolder {root}")
    if not root.exists():
        print("ERROR - Directory does not exist")
        return
    for size in expected_sizes:
        subfolder = root / size
        if not subfolder.exists():
            print(f"ERROR - {size}/  — MISSING")
            continue
        parquets = list(subfolder.glob("*.parquet"))
        total_mb  = sum(f.stat().st_size for f in parquets) / 1e6
        status = "OK" if parquets else "WARNING - no parquet files"
        print(f"   {status}  {size}/  →  {len(parquets)} parquet file(s), {total_mb:.1f} MB")

check_data_folder(REAL_DIR, REAL_EXPECTED_SIZES)
check_data_folder(SYN_DIR,  SYN_EXPECTED_SIZES)


Folder ..\data\benchmark_real
   OK  1M/  →  2 parquet file(s), 172.8 MB
   OK  10M/  →  14 parquet file(s), 1545.3 MB
   OK  50M/  →  66 parquet file(s), 7506.2 MB

Folder ..\data\benchmark_syn
   OK  1M/  →  1 parquet file(s), 95.0 MB
   OK  10M/  →  9 parquet file(s), 952.2 MB
   OK  50M/  →  43 parquet file(s), 4761.3 MB
   OK  5GB/  →  6 parquet file(s), 645.1 MB
   OK  10GB/  →  12 parquet file(s), 1290.9 MB
   OK  20GB/  →  23 parquet file(s), 2581.3 MB


## 9. Summary Statistics Preview

Quick aggregated view across all loaded results (`status == 'ok'` only).

In [16]:
summary = (
    ok_results
    .groupby(["framework", "workload", "dataset_size"])
    .agg(
        time_mean_s        = ("time_s",             "mean"),
        time_std_s         = ("time_s",             "std"),
        peak_mem_mean_mb   = ("peak_memory_mb",     "mean"),
        throughput_mean    = ("throughput_rows_per_s", "mean"),
        n_runs             = ("run_index",           "count"),
    )
    .reset_index()
    .round(3)
)

print(f"Summary over {len(ok_results)} valid rows → {len(summary)} aggregated entries\n")
summary

Summary over 432 valid rows → 96 aggregated entries



,framework,workload,dataset_size,time_mean_s,time_std_s,peak_mem_mean_mb,throughput_mean,n_runs
0,dask,filter,10GB,5.443,0.170,10100.483,2491732.667,3
1,dask,filter,10M,3.930,0.246,7190.712,2553301.333,6
2,dask,filter,1M,1.043,0.501,1024.582,1178333.500,6
3,dask,filter,20GB,17.796,2.306,10683.107,1541916.000,3
4,dask,filter,50M,49.774,17.858,9537.317,1126695.667,6
...,...,...,...,...,...,...,...,...
91,polars_lazy,pipeline,10M,16.242,2.782,8853.738,630911.333,6
92,polars_lazy,pipeline,1M,10.480,0.275,4165.905,95475.500,6
93,polars_lazy,pipeline,20GB,28.880,0.842,12083.857,939200.333,3
94,polars_lazy,pipeline,50M,68.141,26.926,12200.767,842707.667,6


In [17]:
# ── Quick sanity: are time values plausible (not 0 or negative)? ──
bad_times = ok_results[ok_results["time_s"] <= 0]
if bad_times.empty:
    print("All time_s values are positive.")
else:
    print(f"ERROR - {len(bad_times)} row(s) with non-positive time_s:")
    print(bad_times[["framework", "workload", "dataset_size", "run_index", "time_s"]].to_string(index=False))

# ── Quick sanity: memory values ──
bad_mem = ok_results[ok_results["peak_memory_mb"] <= 0]
if bad_mem.empty:
    print("All peak_memory_mb values are positive.")
else:
    print(f"ERROR - {len(bad_mem)} row(s) with non-positive peak_memory_mb")

All time_s values are positive.
All peak_memory_mb values are positive.


---

## Setup Complete

If all checks above passed, the pipeline is ready for analysis.

**Next steps:**
- `data_prep/01a_verify_real_data.ipynb` — explore real dataset characteristics
- `data_prep/01b_calibrate_synthetic.ipynb` — fit synthetic generator parameters
- `data_prep/01c_validate_synthetic.ipynb` — validate synthetic vs real distributions

**Global rules reminder (apply to all notebooks):**
- Never run benchmarks from notebooks
- Always filter `status == 'ok'` before analysis
- Always aggregate over runs (mean ± std)
- Never mix Track A (row-based) and Track B (memory-based) data